##Raw Data Generator
This notebook downloads the data generator from github, it compiles it and it generates the data into a Unity Catalog Volume. 

**Parameters:**
- catalog: the Unity Catalog catalog to be used for this experiment. If the catalog provided does not exist, it will be created automatically. 
- schema: the schema to be used for this experiment. If the schema does not exist, it will be created automatically.
- scale_GB: the scale of the experiment in Gigabyte. We do not recommend to go significantly below 1TB as the cost-savings results will be less evident. **Default: 1TB**


###Sharing Raw Data with Snowflake

Before running the data Generation, you need to think how Snowflake will be able to access the same dataset you will be generating. Here are three strategies: 
- copy the data manually: create a staging in Snowflake pointing to an S3 location and manually copy the data from the Databricks volume to the S3 location. In Unity Catalog, you can select the volume and click on "Details". You will be able to see the Storage Location. A data copy can be issued using the aws cli. 
- Provide credentials to Snowflake to access the specific location in S3 the Databricks volume is storing data into, and create a staging that points to the same location. No data copy is necessary!
- Configure the credentials on Unity Catalog to access a shared S3 location and create the volume pointing to that location. This way the location is already shared and no data copy is necessary. To do this, you need to create a storage credential and set the path when creating the volume. 

The next cell below has an example of how to create the staging in Snowflake, according to the data format that will be generated with this notebook. 

--- Creating a Staging area in Snowflake: Example ---
--- Please Note: the following code will only run on Snowflake. Do not uncomment it in here. ---
/*
CREATE OR REPLACE FILE FORMAT csv_format
  TYPE = csv
  FIELD_DELIMITER = '|'
  NULL_IF = ('NULL', 'null')
  EMPTY_FIELD_AS_NULL = true
  error_on_column_count_mismatch=false;
  
CREATE OR REPLACE STAGE tpcds_stage
  STORAGE_INTEGRATION = s3_int
  URL = 's3://tpcds-datasets/tpcds-2.13/tpcds_sf1000_text/'
  FILE_FORMAT = csv_format;
  */

In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Create Widgets (text) ---

dbutils.widgets.text("dbx_catalog", "", "Databricks Catalog")
dbutils.widgets.text("dbx_schema", "", "Databricks Schema")
dbutils.widgets.text("dbx_volume", "", "Volume name")
dbutils.widgets.text("dbx_volume_location", "", "S3 Location of Volume (Optional)")
dbutils.widgets.text("scale", "1000", "Scale (GB)")

In [ ]:
import os
# --- Retrieve Values, Set Env Vars, and Build Dictionary ---

params = dbutils.widgets.getAll()

for k,v in params.items():
    os.environ[f"W_{k.upper()}"] = v

# --- Validation Check (only required fields) ---

if params["dbx_catalog"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_catalog' is required but is empty.")

if params["dbx_schema"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_schema' is required but is empty.")

if params["dbx_volume"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_volume' is required but is empty.")

if params["scale"] == "":
    raise RuntimeError("Validation Error: Parameter 'scale' is required but is empty.")

In [ ]:
import re

def is_valid_s3_path(path):
    pattern = r"^s3a?://[a-zA-Z0-9.\-_]+(/.*)?$"
    return re.match(pattern, path) is not None

if params['dbx_volume_location'] and params['dbx_volume_location'] != "":
    if not is_valid_s3_path(params['dbx_volume_location']):
        raise Exception(f"Invalid S3 path: {params['dbx_volume_location']}")

In [ ]:
# Verify catalog, schema, and volume exist (don't try to create — they are pre-provisioned infrastructure)
try:
    spark.sql(f"DESCRIBE CATALOG {params['dbx_catalog']}")
    print(f"Catalog OK: {params['dbx_catalog']}")
except Exception as e:
    raise RuntimeError(f"ERROR: Catalog '{params['dbx_catalog']}' not accessible: {e}")

try:
    spark.sql(f"DESCRIBE SCHEMA {params['dbx_catalog']}.{params['dbx_schema']}")
    print(f"Schema OK: {params['dbx_catalog']}.{params['dbx_schema']}")
except Exception as e:
    raise RuntimeError(f"ERROR: Schema '{params['dbx_catalog']}.{params['dbx_schema']}' not accessible: {e}")

try:
    spark.sql(f"DESCRIBE VOLUME {params['dbx_catalog']}.{params['dbx_schema']}.{params['dbx_volume']}")
    print(f"Volume OK: {params['dbx_catalog']}.{params['dbx_schema']}.{params['dbx_volume']}")
except Exception as e:
    raise RuntimeError(f"ERROR: Volume '{params['dbx_catalog']}.{params['dbx_schema']}.{params['dbx_volume']}' not accessible: {e}")

In [ ]:
%sh
apt-get update
apt-get -y install gcc make flex bison byacc git

In [ ]:
%sh 
mkdir -p /mnt/dev
cd /mnt/dev

git clone https://github.com/databricks/tpcds-kit.git

In [ ]:
%sh 
cd /mnt/dev/tpcds-kit/tools
make OS=LINUX

In [ ]:
import os, re, subprocess, tempfile, shutil

# --- PySpark Distributed Data Generation ---
TPCDS_KIT_DIR = "/mnt/dev/tpcds-kit/tools"
DSDGEN_BIN = os.path.join(TPCDS_KIT_DIR, "dsdgen")
TPCDS_IDX = os.path.join(TPCDS_KIT_DIR, "tpcds.idx")
SCALE = int(params["scale"])
VOLUME_ROOT = f"/Volumes/{params['dbx_catalog']}/{params['dbx_schema']}/{params['dbx_volume']}"
VOLUME_PATH = f"{VOLUME_ROOT}/{SCALE}"
N_PARTS = max(2, SCALE // 10)  # ~10 GB per child, minimum 2 (dsdgen requirement)

print(f"Scale: {SCALE} GB  |  Children: {N_PARTS}  |  Output: {VOLUME_PATH}/")

# Check if data already exists for this scale
_existing_tables = []
try:
    for item in dbutils.fs.ls(VOLUME_PATH):
        if item.isDir():
            # Check if directory has .dat files
            try:
                files = [f for f in dbutils.fs.ls(item.path) if f.name.endswith(".dat")]
                if files:
                    _existing_tables.append(item.name.rstrip("/"))
            except:
                pass
except:
    pass

if len(_existing_tables) >= 20:
    print(f"\nDataset already exists at {VOLUME_PATH}/ with {len(_existing_tables)} tables:")
    for t in sorted(_existing_tables)[:5]:
        print(f"  {t}/")
    print(f"  ... and {len(_existing_tables) - 5} more")
    print("\nSkipping data generation. To force regeneration, delete the volume contents first.")
    dbutils.notebook.exit(f"SKIPPED: dataset already exists ({len(_existing_tables)} tables)")

# Distribute dsdgen binary and index file to all Spark workers
sc.addFile(DSDGEN_BIN)
sc.addFile(TPCDS_IDX)

# Tables to generate (returns tables are co-generated with their parent sales tables)
TABLES_TO_GENERATE = [
    "call_center", "catalog_page", "catalog_sales", "customer",
    "customer_address", "customer_demographics", "date_dim", "dbgen_version",
    "household_demographics", "income_band", "inventory", "item", "promotion",
    "reason", "ship_mode", "store", "store_sales", "time_dim", "warehouse",
    "web_page", "web_sales", "web_site",
]

# Co-generated tables that also produce output files
ALL_TABLE_DIRS = TABLES_TO_GENERATE + ["catalog_returns", "store_returns", "web_returns"]

# Pre-create all subdirectories on the driver (workers cannot mkdir on /Volumes/ FUSE)
for tbl in ALL_TABLE_DIRS:
    os.makedirs(os.path.join(VOLUME_PATH, tbl), exist_ok=True)

# Pattern to extract table name from dsdgen filenames (e.g. "store_returns_4_16.dat" -> "store_returns")
_table_name_re = re.compile(r"_\d+_\d+\.dat$")

# Capture variables for closure serialization
_scale = SCALE
_n_parts = N_PARTS
_volume_path = VOLUME_PATH

def generate_table(table_name):
    """Distribute dsdgen child processes across Spark workers for a single table."""
    _table = table_name
    child_ids = list(range(1, _n_parts + 1))
    rdd = sc.parallelize(child_ids, len(child_ids))

    def _run_child(iterator):
        from pyspark import SparkFiles
        import os, re, subprocess, tempfile, shutil

        dsdgen_bin = SparkFiles.get("dsdgen")
        dist_file = SparkFiles.get("tpcds.idx")
        if not os.access(dsdgen_bin, os.X_OK):
            os.chmod(dsdgen_bin, 0o755)

        table_re = re.compile(r"_\d+_\d+\.dat$")

        for child_id in iterator:
            tmpdir = tempfile.mkdtemp(prefix="tpcds_")
            try:
                os.symlink(dsdgen_bin, os.path.join(tmpdir, "dsdgen"))
                os.symlink(dist_file, os.path.join(tmpdir, "tpcds.idx"))

                cmd = [
                    "./dsdgen",
                    "-SCALE", str(_scale),
                    "-TABLE", _table,
                    "-DIR", ".",
                    "-FORCE",
                    "-PARALLEL", str(_n_parts),
                    "-CHILD", str(child_id),
                ]
                proc = subprocess.run(cmd, capture_output=True, cwd=tmpdir, timeout=7200)
                if proc.returncode != 0:
                    raise RuntimeError(
                        f"dsdgen child {child_id} for {_table} failed: "
                        f"{proc.stderr.decode('utf-8', errors='replace')}"
                    )

                for fname in os.listdir(tmpdir):
                    if fname.endswith(".dat"):
                        tbl_name = table_re.sub("", fname)
                        tbl_dir = os.path.join(_volume_path, tbl_name)
                        shutil.copy2(
                            os.path.join(tmpdir, fname),
                            os.path.join(tbl_dir, fname),
                        )
                        yield fname
            finally:
                shutil.rmtree(tmpdir, ignore_errors=True)

    return rdd.mapPartitions(_run_child).collect()

# Generate all tables distributed across the cluster
for table in TABLES_TO_GENERATE:
    print(f"Generating {table} ...")
    files = generate_table(table)
    print(f"  -> {len(files)} file(s) generated")

print("All data generation processes have completed.")